# 09 · 本地影像／ROI：真实文件、几何质控与原生 55 项特征

本页读取你提供的本机样例，不下载、不合成影像或标签、不调用商业平台运行时。原始 NIfTI、mask 与本次执行输出均留在仓库外。目录中未找到整体再分发许可证，本页执行结果不用于公共站点、wheel 或 sdist。

输入目录称为 CT，但未附强度标定协议，因此这里不将数值自动解释为 HU。保留完整 RAS affine；展示的是源数组的 z 索引切面，不标注成已经重定向的解剖轴位。

## 1. 检查真正运行的环境

所有提取由 `ov_radiomics` Rust 核心执行；这里明确记录解释器、扩展 SHA 和源码实现身份。

In [ ]:
from pathlib import Path
import hashlib, json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import ov_radiomics as rad
from ov_radiomics._backend import resolve_backend

if 'TUTORIAL_DATA_ROOT' not in globals() or 'TUTORIAL_RUN_ROOT' not in globals():
    raise RuntimeError('请用本教程的 run_local_sample_tutorials.py --data-root ... --output ... 执行；或先设置这两个路径变量。')
DATA = Path(TUTORIAL_DATA_ROOT).resolve() / 'CT'
output = Path(TUTORIAL_RUN_ROOT).resolve()
output.mkdir(parents=True, exist_ok=False)
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
def save_json(name, value):
    (output / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
native = resolve_backend('rust')
runtime = dict(backend=rad.backend_info(), interpreter=sys.executable, python=sys.version,
    package_version=rad.__version__, package_path=rad.__file__, native_sha256=sha256(native.__file__),
    implementation_sha256=native.IMPLEMENTATION_SHA256, data_kind='local-image-mask-tabular-no-clinical-validity',
    distribution='local-only: source/redistribution license not established')
assert runtime['backend']['metrics_backend'] == 'rust'
save_json('tutorial-runtime.json', runtime)
display(pd.Series(runtime['backend'], name='actual backend'))

## 2. 冻结输入清单和分析参数

按同名文件精确配对，不按目录顺序猜配对。`label_multicenter.csv` 的文件名不证明真实多中心；这里仅保留其提供的 train／val 分组，不重新挑划分。

提取输入清单中全部图像的 label 1；首次本地验收使用了237对文件。参数预先固定为 `python_golden_v1`、fixed-count 32、distance 1／direction_mean、GLDM alpha 0、voxel shift 0。不做图像归一化、裁剪、截断或重采样。32 bins 是软件教学选择，不是临床 CT preset。

部分 affine 被原生读取器标为 shear，因此本页仅取 16 项一阶＋39 项纹理（55 项）；不调用要求正交网格的 11 项 `shape_basic`。这不是把提供的 107 列旧特征表重新提取了一遍。

In [ ]:
provided_split = pd.read_csv(DATA / 'label_multicenter.csv', dtype=str)
provided_labels = pd.read_csv(DATA / 'label.csv', dtype=str)
assert provided_split.ID.is_unique and provided_labels.ID.is_unique
assert set(provided_split.group) == {'train', 'val'}
assert set(provided_split.label) == {'0', '1'}
pd.testing.assert_series_equal(provided_split.set_index('ID').label.sort_index(),
    provided_labels.set_index('ID').label.sort_index())
images = {p.name: p for p in (DATA / 'images').glob('*.nii.gz')}
masks = {p.name: p for p in (DATA / 'masks').glob('*.nii.gz')}
assert set(images) == set(masks) == set(provided_split.ID)
ids = sorted(images, key=lambda x: (int(x.split('.')[0]), x))
families = ['firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm']
config = dict(profile='python_golden_v1', discretization=dict(method='fixed_count', count=32),
    aggregation='direction_mean', gldm_alpha=0, voxel_array_shift=0.)
protocol = dict(config=config, families=families, roi_label=1, threads=1,
    image_processing='none', alignment_policy='native validation; explicit nearest-neighbor mask-to-image resampling only for equal-shape millimeter grids with <=0.001mm corner drift',
    maximum_permitted_header_drift_mm=0.001, physical_shape_features='not requested',
    patient_identity='source ID grouping only; independent patients are not established')
save_json('protocol.json', protocol)
source_hashes = {str(p): sha256(p) for p in
    [DATA / 'label.csv', DATA / 'label_multicenter.csv'] +
    [p for identity in ids for p in (images[identity], masks[identity])]}
save_json('source-hashes.json', source_hashes)
display(provided_split.groupby(['group', 'label']).size().unstack(fill_value=0))
print('Exact image/mask pairs:', len(ids))
display(protocol)

## 3. 看清原始拒绝，再显式处理微小网格差异

首例的 image 使用 sform，mask 使用 qform。虽然 shape 一样，网格外角点有微小物理坐标差异，原生严格校验会拒绝，不能直接把 mask 的 affine 强行改成 image affine。

本页只允许同 shape、毫米单位且最大角点漂移不超过 0.001 mm 的样例进入**显式最近邻 mask 重采样到 image 网格**。这不是放宽包内的校验容差：处理后仍由原生 `ImageSample` 正常验证。漂移更大、shape 不同或 ROI 消失会停止并保留错误，不自动注册或居中。

下面保留首例的真实拒绝信息，之后的表逐例记录是否重采样、改变了多少标签、ROI 前后体素数及原始警告。image 强度完全不变；原始 mask 文件也不改写。

In [ ]:
first_image = rad.image.read_nifti(images[ids[0]])
first_mask = rad.mask.read_nifti(masks[ids[0]])
try:
    rad.ImageSample(dict(sample_id=ids[0], patient_id=ids[0]), first_image, first_mask)
    first_rejection = None
except ValueError as error:
    first_rejection = str(error)
print('Original pair validation:', first_rejection or 'passed')
display(first_image.geometry)
display(first_mask.geometry)
del first_image, first_mask

def corner_drift(image_geometry, mask_geometry):
    assert image_geometry['shape'] == mask_geometry['shape']
    assert image_geometry['unit'] == mask_geometry['unit'] == 'millimeter'
    shape = image_geometry['shape']
    corners = np.asarray([[x, y, z, 1.] for x in [-.5, shape[0]-.5]
        for y in [-.5, shape[1]-.5] for z in [-.5, shape[2]-.5]])
    first = corners @ np.asarray(image_geometry['affine']).T
    second = corners @ np.asarray(mask_geometry['affine']).T
    return float(np.linalg.norm((first-second)[:, :3], axis=1).max())

## 4. 全部文件逐例提取与可追溯质控

执行后看实际成功数，不能只展示一个能跑的例子。几何 warnings 原样记录；没有因 warnings 就关闭校验。`patient_id` 暂以 source ID 作为记录分组键，仅用于 API 合约，不能据此推断生物学患者独立。

原始几何警告保存在独立 QA 中；本页不将其缺省为“数据完美”。缺失特征保持 NaN，正负无穷会报错。

In [ ]:
rows, qa, failures, previews = [], [], [], []
started = time.perf_counter()
for index, identity in enumerate(ids):
    try:
        image = rad.image.read_nifti(images[identity])
        source_mask = rad.mask.read_nifti(masks[identity])
        drift = corner_drift(image.geometry, source_mask.geometry)
        original_labels = source_mask.array
        if set(np.unique(original_labels)) - {0, 1}:
            raise ValueError('This frozen local tutorial expects explicit binary mask labels 0/1.')
        if not (original_labels == 1).any():
            raise ValueError('Source ROI label 1 is empty.')
        metadata = dict(sample_id=identity, patient_id=identity)
        rejection = None
        try:
            sample = rad.ImageSample(metadata, image, source_mask)
            aligned_mask = source_mask
            correction = 'none'
        except ValueError as error:
            rejection = str(error)
            if 'world-coordinate grids differ' not in rejection:
                raise
            if drift > protocol['maximum_permitted_header_drift_mm']:
                raise ValueError('Grid drift exceeds the declared small-header-drift policy.') from error
            aligned_mask = rad.mask.resample_like(source_mask, image)
            sample = rad.ImageSample(metadata, image, aligned_mask)
            correction = 'explicit_nearest_mask_resample_to_image'
        aligned_labels = aligned_mask.array
        if not (aligned_labels == 1).any():
            raise ValueError('ROI label 1 disappeared after explicit mask resampling.')
        table = sample.extract(config, families=families, threads=1)
        assert table.features.shape == (1, 56)
        rows.append(table.features)
        record = dict(ID=identity, max_outer_corner_drift_mm=drift,
            alignment_action=correction, original_rejection=rejection,
            original_ROI_voxels=int((original_labels==1).sum()),
            aligned_ROI_voxels=int((aligned_labels==1).sum()),
            changed_mask_voxels=int(np.count_nonzero(original_labels!=aligned_labels)),
            image_warning_codes=';'.join(x['code'] for x in image.geometry['diagnostics'] if x['severity']=='warning'),
            mask_warning_codes=';'.join(x['code'] for x in source_mask.geometry['diagnostics'] if x['severity']=='warning'),
            voxel_volume_mm3=float(abs(np.linalg.det(np.asarray(image.geometry['affine'])[:3,:3]))))
        qa.append(record)
        if index < 3:
            chosen = aligned_labels == 1
            z = int(np.argmax(chosen.sum(axis=(1,2))))
            plane = image.array[z]
            roi_values = image.array[chosen]
            lo, hi = np.percentile(roi_values, [1, 99])
            previews.append((identity, z, plane.copy(), chosen[z].copy(), float(lo), float(hi)))
        del image, source_mask, aligned_mask, sample, original_labels, aligned_labels
    except Exception as error:
        failures.append(dict(ID=identity, error_type=type(error).__name__, error=str(error)))
    (output / 'progress.json').write_text(json.dumps(dict(processed=index+1, total=len(ids),
        success=len(rows), failures=len(failures))), encoding='utf-8')
    if (index+1)%25 == 0:
        print('Processed:', index+1, '/', len(ids), '; failures:', len(failures))
pd.DataFrame(failures, columns=['ID','error_type','error']).to_csv(output/'failures.csv', index=False)
save_json('failures.json', failures)
qa = pd.DataFrame(qa)
qa.to_csv(output/'qa.csv', index=False)
assert not failures, 'Data failures retained in failures.csv; resolve them explicitly before modeling.'
features = pd.concat(rows, ignore_index=True)
assert features.ID.is_unique and set(features.ID) == set(ids)
assert len(features.columns) == 56
assert not np.isinf(features.drop(columns='ID').to_numpy()).any()
features.to_csv(output/'features.csv', index=False)
display(features[['ID','original_firstorder_Mean','original_firstorder_Variance']].head())
display(qa[['max_outer_corner_drift_mm','original_ROI_voxels','aligned_ROI_voxels','changed_mask_voxels']].describe())
display(qa.alignment_action.value_counts())
print('Rows:', len(features), '; native features:', len(features.columns)-1)
print('Undefined feature cells preserved:', int(features.drop(columns='ID').isna().sum().sum()))
print('Elapsed seconds (execution evidence, not a benchmark):', time.perf_counter()-started)

## 5. 解释实际图像和结果

红线是实际 ROI 的边界。显示范围采用该例 ROI 强度的 1%–99% 分位数，仅改变显示，不改变提取数据。图中切面采用源数组 z 索引；没有擅自标注 RAS 解剖左右。

几何表中的体素体积为 affine 三阶行列式绝对值的 QA 数值，不新增 `shape_basic` 特征，也不把近似 shear 当作已经正交。

In [ ]:
fig, axes = plt.subplots(len(previews), 2, figsize=(10, 4*len(previews)), squeeze=False, constrained_layout=True)
for number, (identity, z, plane, roi, lo, hi) in enumerate(previews):
    for ax in axes[number]:
        ax.imshow(plane, cmap='gray', origin='lower', vmin=lo, vmax=hi)
        ax.set(xlabel='Source voxel x index', ylabel='Source voxel y index')
    axes[number,0].set_title(f'Local record {number+1}, source z={z}')
    axes[number,1].contour(roi.astype(float), levels=[.5], colors=['#e45756'], linewidths=1.)
    axes[number,1].set_title('Actual ROI label 1')
fig.suptitle('Index-space display; full affine and QA retained; display window only')
fig.savefig(output/'roi-previews.png', dpi=140)
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
axes[0].hist(qa.aligned_ROI_voxels, bins=25, color='#2f6b8a')
axes[0].set(xlabel='Label-1 voxel count', ylabel='Source records', title='Observed ROI sizes')
axes[1].hist(features.original_firstorder_Mean, bins=25, color='#d08b41')
axes[1].set(xlabel='Original first-order Mean (input units)', ylabel='Source records', title='Native Mean values')
fig.savefig(output/'feature-overview.png', dpi=140)
plt.show()

## 6. 验证原始输入没有变动

所有来源 SHA 再次核对。Notebook、图、特征和本机路径收据均只保存在这次外置执行目录。下一篇使用相同记录身份与提供的固定 train／val 继续建模。

In [ ]:
assert all(sha256(path) == value for path, value in source_hashes.items())
receipt = dict(status='passed', source_records=len(ids), extracted_records=len(features),
    feature_count=55, shape_basic_requested=False, protocol=protocol,
    resampled_mask_records=int(qa.alignment_action.ne('none').sum()),
    records_with_changed_mask_labels=int(qa.changed_mask_voxels.gt(0).sum()),
    max_outer_corner_drift_mm=float(qa.max_outer_corner_drift_mm.max()),
    warnings_retained=True, original_input_hashes_unchanged=True, outcomes_used=False,
    clinical_validity_claim=False, distribution='local-only', runtime=runtime,
    outputs={name:sha256(output/name) for name in ['features.csv','qa.csv','failures.csv','roi-previews.png','feature-overview.png']})
save_json('receipt.json', receipt)
display(pd.Series({k:v for k,v in receipt.items() if k in ['source_records','extracted_records','feature_count','resampled_mask_records','records_with_changed_mask_labels','max_outer_corner_drift_mm']}))